# Lab 1 - Microstates of a system of coins

Runs the code in `src/microstates.py`, its unit tests, and `main.py`. Use **Run All** in VS Code or Jupyter Notebook.

## Setup
Adds the repository root directory to Python's `sys.path` so that modules inside `src/` can be imported seamlessly.

In [ ]:
import os
import sys
import subprocess
from math import comb
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "src").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np

from src.microstates import (
    generate_microstates_baseline,
    generate_microstates_bitwise,
    measure_microstates_optimized,
)

print("Repo root:", ROOT)

## Unit Tests
Executes `pytest` on the `tests/` directory to verify all test cases pass.

In [ ]:
result = subprocess.run([sys.executable, "-m", "pytest", "-v"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout if result.stdout else result.stderr)

## Bitwise Encoding Mechanics
Each integer $s \in [0, 2^N)$ represents one unique microstate. Bit $i$ indicates coin $i$ ($1 = \text{Heads}$, $0 = \text{Tails}$).

In [ ]:
N = 3
for s in range(1 << N):
    binary_str = f"{s:0{N}b}"
    coins = "".join(['H' if bit == '1' else 'T' for bit in binary_str])
    heads = bin(s).count('1')
    print(f"s = {s}  binary = {binary_str}  coins = {coins}  heads = {heads}")

## 1a: Microstate Breakdown for $N = 7$ Coins ($2^7 = 128$ states)

In [ ]:
N = 7
groups = {}
for s in range(1 << N):
    heads = bin(s).count('1')
    groups.setdefault(heads, []).append(s)

print(f"{'macrostate':<22}{'# microstates':>14}")
for k in sorted(groups.keys(), reverse=True):
    label = f"{k} heads, {N-k} tails"
    print(f"{label:<22}{len(groups[k]):>14}")
print(f"{'total':<22}{sum(len(v) for v in groups.values()):>14}")

## Verification: Binomial Coefficients $\Omega(N,k) = \binom{N}{k}$

In [ ]:
for N in [5, 7, 10, 16]:
    counts = [0] * (N + 1)
    for s in range(1 << N):
        counts[bin(s).count('1')] += 1
    expected = [comb(N, k) for k in range(N + 1)]
    print(f"N = {N:2d}: {counts}  matches comb: {counts == expected}  sum = 2^{N}: {sum(counts) == 2**N}")

## 1b: Performance Benchmarking Across Methods

In [ ]:
print(f"{'N':<4} | {'2^N Microstates':<16} | {'Method':<22} | {'Time (s)':<12} | {'RAM (MiB)':<10}")
print("-" * 72)

methods = [
    ("Method 1 (Itertools)", generate_microstates_baseline),
    ("Method 2 (Bitwise Gen)", generate_microstates_bitwise),
    ("Method 3 (Optimized)", measure_microstates_optimized),
]

for N in range(10, 22, 4):
    for name, func in methods:
        count, exec_time, ram_mb = func(N)
        print(f"{N:<4} | {count:<16} | {name:<22} | {exec_time:<12.6f} | {ram_mb:<10.4f}")
    print("-" * 72)

## 1b: Time & Memory Scaling Analysis

In [ ]:
ns, times, rams = [], [], []
for N in range(10, 26, 2):
    count, exec_time, ram_mb = measure_microstates_optimized(N)
    ns.append(N)
    times.append(exec_time)
    rams.append(ram_mb)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(ns, times, "o-", color="tab:blue")
ax1.set_yscale("log")
ax1.set_xlabel("Number of coins $N$")
ax1.set_ylabel("Execution Time (s)")
ax1.set_title("Log Execution Time vs N")
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot(ns, rams, "s-", color="tab:red")
ax2.set_xlabel("Number of coins $N$")
ax2.set_ylabel("Peak RAM (MiB)")
ax2.set_title("Memory Footprint vs N")
ax2.grid(True, linestyle="--", alpha=0.6)

fig.tight_layout()
plt.show()

## Execution of `main.py`
Runs the main project entry point and displays standard output.

In [ ]:
result = subprocess.run([sys.executable, "main.py"], cwd=ROOT, capture_output=True, text=True)
print(result.stdout if result.stdout else result.stderr)